In [3]:
%pip install azure-search-documents==11.4.0 openai

  Using cached azure_common-1.1.28-py2.py3-none-any.whl.metadata (5.0 kB)
  Using cached isodate-0.7.2-py3-none-any.whl.metadata (11 kB)
Using cached azure_common-1.1.28-py2.py3-none-any.whl (14 kB)
Using cached isodate-0.7.2-py3-none-any.whl (22 kB)

   ------------- -------------------------- 1/3 [isodate]
   -------------------------- ------------- 2/3 [azure-search-documents]
   -------------------------- ------------- 2/3 [azure-search-documents]
   -------------------------- ------------- 2/3 [azure-search-documents]
   -------------------------- ------------- 2/3 [azure-search-documents]
   -------------------------- ------------- 2/3 [azure-search-documents]
   -------------------------- ------------- 2/3 [azure-search-documents]
   -------------------------- ------------- 2/3 [azure-search-documents]
   ---------------------------------------- 3/3 [azure-search-documents]

Note: you may need to restart the kernel to use updated packages.


In [5]:
# Load environment variables from .env file
from azure.identity import DefaultAzureCredential
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv
from openai import AzureOpenAI
import os

load_dotenv(override=True)

# Azure AI Search
search_endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
index_name = os.getenv("AZURE_SEARCH_INDEX", "rag-1776332199094")
search_api_key = os.getenv("AZURE_SEARCH_API_KEY") or os.getenv("AZURE_SEARCH_ADMIN_KEY")

# Prefer API key to avoid RBAC issues during local notebooks.
if search_api_key:
    credential = AzureKeyCredential(search_api_key)
    search_auth_mode = "API key"
else:
    credential = DefaultAzureCredential()
    search_auth_mode = "DefaultAzureCredential (RBAC)"

# Azure OpenAI (embeddings)
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
if not azure_openai_endpoint:
    # Optional fallback if you have Foundry-style env vars
    foundry_base = os.getenv("AZURE_FOUNDRY_OPENAI_BASE_URL", "")
    if "/openai" in foundry_base:
        azure_openai_endpoint = foundry_base.split("/openai")[0]

azure_openai_api_key = os.getenv("AZURE_OPENAI_API_KEY")
embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002")

if not azure_openai_endpoint:
    raise ValueError("Missing AZURE_OPENAI_ENDPOINT in .env")
if not azure_openai_api_key:
    raise ValueError("Missing AZURE_OPENAI_API_KEY in .env")

client = AzureOpenAI(
    azure_endpoint=azure_openai_endpoint,
    api_key=azure_openai_api_key,
    api_version="2024-02-01"
 )

print(f"Using Azure Search endpoint: {search_endpoint}")
print(f"Using Azure Search index: {index_name}")
print(f"Azure Search auth mode: {search_auth_mode}")
print(f"Using embedding deployment: {embedding_deployment}")

Using Azure Search endpoint: https://ai-search-mario2.search.windows.net
Using Azure Search index: rag-1776332199094
Azure Search auth mode: API key
Using embedding deployment: text-embedding-ada-002






## PARTE 2 — Búsquedas prácticas (ejecución en Python)

Entregar un script o notebook (.ipynb recomendado) con ejecuciones de los siguientes tipos de búsqueda y ejemplos de resultados (top-5):

1. Vector Search
2. Hybrid Search (vector + keyword)
3. Semantic Search (query_type=semantic)
4. Semantic Hybrid Search (semantic + vector)

[Referencia para semantic ranking](https://github.com/Azure-Samples/azure-search-python-samples/blob/main/Quickstart-Semantic-Ranking/semantic-ranking-quickstart.ipynb)
[Referencia para vector search](https://github.com/Azure-Samples/azure-search-python-samples/blob/main/Quickstart-Vector-Search/vector-search-quickstart.ipynb)


### Vector Search

In [8]:
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
from textwrap import fill, indent

# Inicializamos el cliente
search_client = SearchClient(search_endpoint, index_name, credential)
query = "¿Quién se clasificó en el enfrentamiento entre FC Barcelona y Atlético de Madrid?"

# 1) Generar vector con Azure OpenAI
response = client.embeddings.create(
    input=query,
    model=embedding_deployment
)
embedding = response.data[0].embedding

# 2) Construir la query vectorial
vector_query = VectorizedQuery(
    vector=embedding,
    k_nearest_neighbors=5,
    fields="text_vector" # Asegúrate de que este es el nombre correcto en tu índice
)

# 3) Ejecutar búsqueda
results = search_client.search(
    search_text=None,
    vector_queries=[vector_query],
    select=["title", "chunk"],
    top=5
)

rows = list(results)

# ---------------------------------------------------------
# OUTPUT FORMATEADO (ESTILO PROFESIONAL PARA NOTEBOOK)
# ---------------------------------------------------------
print("╔══════════════════════════════════════════════════════════════════════════════════════╗")
print("║                           🔍 RESULTADOS DE BÚSQUEDA VECTORIAL                        ║")
print("╚══════════════════════════════════════════════════════════════════════════════════════╝")
print(f" 🎯 Búsqueda: {query}")
print(f" 📂 Índice:   {index_name}")
print(f" 📊 Matches:  {len(rows)} fragmentos recuperados")
print("────────────────────────────────────────────────────────────────────────────────────────")

for i, result in enumerate(rows, start=1):
    title = result.get("title") or "(sin título)"
    chunk = (result.get("chunk") or "").strip()
    score = float(result.get("@search.score", 0))

    # Limpiamos saltos de línea extraños del PDF y limitamos a 450 caracteres
    chunk_clean = " ".join(chunk.split())
    if len(chunk_clean) > 450:
        chunk_clean = chunk_clean[:450] + "..."
        
    # Formateamos el bloque de texto
    snippet = fill(chunk_clean, width=80)

    print(f" 🔹 RESULTADO #{i}  |  Relevancia (Score): {score:.4f}")
    print(f" 📄 Documento: {title}")
    print(f" 💬 Contenido recuperado:")
    # Identamos el fragmento para que se diferencie visualmente de los metadatos
    print(indent(snippet, "      ")) 
    print("────────────────────────────────────────────────────────────────────────────────────────")

╔══════════════════════════════════════════════════════════════════════════════════════╗
║                           🔍 RESULTADOS DE BÚSQUEDA VECTORIAL                        ║
╚══════════════════════════════════════════════════════════════════════════════════════╝
 🎯 Búsqueda: ¿Quién se clasificó en el enfrentamiento entre FC Barcelona y Atlético de Madrid?
 📂 Índice:   rag-1776332199094
 📊 Matches:  5 fragmentos recuperados
────────────────────────────────────────────────────────────────────────────────────────
 🔹 RESULTADO #1  |  Relevancia (Score): 0.8835
 📄 Documento: RAG Champions League Cuartos 2025_2026.pdf
 💬 Contenido recuperado:
      a la desesperada por forzar la prórroga, el Bayern armó un contragolpe de manual
      en el cuarto minuto de tiempo de descuento (90+4'). Michael Olise, culminando
      una actuación individual soberbia, definió con frialdad para sellar la victoria
      del Bayern por 4-3, confirmando su merecido pase a semifinales y consumando el
      segu

In [9]:
# HYBRID SEARCH = keyword + vector
from azure.search.documents.models import VectorizedQuery
from textwrap import fill

hybrid_query = "¿Quién se clasificó en el enfrentamiento entre FC Barcelona y Atlético de Madrid?"

# 1) Embedding de la consulta (parte vectorial)
hybrid_embedding = client.embeddings.create(
    input=hybrid_query,
    model=embedding_deployment
).data[0].embedding

vector_query = VectorizedQuery(
    vector=hybrid_embedding,
    k_nearest_neighbors=5,
    fields="text_vector"
 )

# 2) Búsqueda híbrida: search_text (keyword) + vector_queries (vector)
hybrid_results = search_client.search(
    search_text=hybrid_query,
    vector_queries=[vector_query],
    select=["title", "chunk"],
    top=5
)

hybrid_rows = list(hybrid_results)
print("=" * 90)
print("HYBRID SEARCH RESULTS")
print("=" * 90)
print(f"Query: {hybrid_query}")
print(f"Resultados devueltos: {len(hybrid_rows)}")
print("=" * 90)

for i, result in enumerate(hybrid_rows, start=1):
    title = result.get("title") or "(sin título)"
    chunk = (result.get("chunk") or "").strip()
    score = float(result.get("@search.score", 0))

    snippet = fill(chunk[:420], width=95) if chunk else "(sin contenido)"

    print(f"[{i:02}] Score: {score:.4f}")
    print(f"Título: {title}")
    print("Fragmento:")
    print(snippet)
    print("-" * 90)

HYBRID SEARCH RESULTS
Query: ¿Quién se clasificó en el enfrentamiento entre FC Barcelona y Atlético de Madrid?
Resultados devueltos: 5
[01] Score: 0.0333
Título: RAG Champions League Cuartos 2025_2026.pdf
Fragmento:
a la desesperada por forzar la prórroga, el Bayern  armó un contragolpe de manual en el cuarto
minuto de tiempo de descuento (90+4'). Michael  Olise, culminando una actuación individual
soberbia, definió con frialdad para sellar la victoria  del Bayern por 4-3, confirmando su
merecido pase a semifinales y consumando el segundo  fracaso consecutivo del Madrid en la
barrera de los cuartos de final.4   El análisis estra
------------------------------------------------------------------------------------------
[02] Score: 0.0300
Título: RAG Champions League Cuartos 2025_2026.pdf
Fragmento:
y la urgencia azulgrana se materializó de  forma fulgurante en un arranque furioso y eléctrico
que hizo soñar legítimamente a su  numerosa afición con la remontada.21 En el minuto 4 de
juego,

### Hybrid Search

### Semantic Search

### Semantic Hybrid Search

## ⭐ Extra (elige al menos UNA opción y documenta)

A. Añadir una **custom skill** (OCR o endpoint propio) al skillset: documentar su implementación y efecto en indexing. Referencia: https://learn.microsoft.com/en-us/azure/search/cognitive-search-defining-skillset

B. Añadir un **scoring profile** al índice y explicar cómo afecta al ranking. Referencia: https://learn.microsoft.com/es-es/azure/search/index-add-scoring-profiles?tabs=python

C. Implementar **multimodal search** (texto + imágenes) si los documentos contienen imágenes. Referencia: https://learn.microsoft.com/en-us/azure/search/multimodal-search-overview

---





## Entregables (qué debe contener la entrega)

1. Un archivo en formato .docx, .ipynb o .md documentando la Parte 1
2. `practica_vector_search.ipynb` o `practica_vector_search.py` con la Parte 2 ejecutada y resultados incluidos.
4. (Opcional) Documentación y explicación de la implementación del extra elegido

---
